# Simulation Tables

This notebook builds exactly **5 tables** from cached `.jbl` files:
1. **Nonparametrics** (`simulations/nonparametric_fit`)
2-5. **Semiparametrics** by DGP (`simulations/semiparametric_cov`)

Displayed values are **mean (SE)**.

In [13]:
from pathlib import Path
import os
import sys
import re
import numpy as np
import pandas as pd
import joblib
from IPython.display import display, Markdown

def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if (cand / 'simulations').exists() and (cand / 'nnpiv').exists():
            return cand
    raise RuntimeError('Could not locate repo root containing simulations/ and nnpiv/.')

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import simulations.dgps_nested as dgps_np
import simulations.dgps_mediated as dgps_sp

# --------------------------------
# Paths & high-level configuration
# --------------------------------
NP_DIR = REPO_ROOT / 'simulations' / 'nonparametric_fit'
SP_DIR = REPO_ROOT / 'simulations' / 'semiparametric_cov'
CONFIG_DIR = REPO_ROOT / 'simulations'
OUTDIR = REPO_ROOT / 'output'  # where to save TeX tables
os.makedirs(OUTDIR, exist_ok=True)
TABLE_TEX_PATH = OUTDIR / 'tables.tex'

BENCH_COLS = ['2SLS', 'series', 'reg.']
NP_FN_ORDER = [('linear', 8), ('pwlinear', 15), ('sigmoid', 2), ('exponential', 16)]
SP_FN_ORDER = [('linear', 0), ('pwlinear', 1), ('sigmoid', 2), ('exponential', 4)]
SP_TRUE_PARAM = 4.05

# Outlier trimming options. Set any value to None to disable that trim.
NP_TRIM_UPPER_Q = 99.0            # nonparametric MSE table
SP_BIAS_TRIM_UPPER_Q = 99.0      # semiparametric bias row
SP_VARIANCE_TRIM_UPPER_Q = 99.0  # semiparametric variance row


def _sample_sd(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size <= 1:
        return np.nan
    return float(np.std(arr, ddof=1))


def _mean_sd_se(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    n = int(arr.size)
    if n == 0:
        return {'mean': np.nan, 'sd': np.nan, 'se': np.nan, 'n': 0}
    sd = _sample_sd(arr)
    se = sd / np.sqrt(n) if n > 1 else np.nan
    return {'mean': float(np.mean(arr)), 'sd': sd, 'se': se, 'n': n}


def _mean_sd_se_coverage(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    n = int(arr.size)
    if n == 0:
        return {'mean': np.nan, 'sd': np.nan, 'se': np.nan, 'n': 0}
    p_hat = float(np.mean(arr))
    sd = float(np.sqrt(p_hat * (1 - p_hat)))
    se = float(np.sqrt(p_hat * (1 - p_hat) / n))
    return {'mean': p_hat, 'sd': sd, 'se': se, 'n': n}


def _fmt_mean_se(mean_val, se_val, digits=4):
    if pd.isna(mean_val):
        return ''
    if pd.isna(se_val):
        return f'{mean_val:.{digits}g}'
    return f'{mean_val:.{digits}g} ({se_val:.{digits}g})'


def _ordered_unique(values):
    out = []
    seen = set()
    for value in values:
        if value is None or value in seen:
            continue
        out.append(value)
        seen.add(value)
    return out


def _method_keys_from_config(path: Path):
    text = path.read_text(encoding='utf-8')
    m = re.search(r'["\']methods["\']\s*:\s*\{(?P<body>.*?)\n\s*\}', text, flags=re.S)
    if m is None:
        return []
    return re.findall(r'["\']([^"\']+)["\']\s*:', m.group('body'))


def _profile_from_config(path: Path, prefix: str):
    return path.stem.replace(f'config_{prefix}_', '', 1)


def _display_method(profile: str, raw_method: str):
    profile = str(profile)
    raw_method = str(raw_method)

    if profile == 'benchmark' and raw_method == '2SLS':
        return '2SLS'
    if profile == 'benchmark2' and raw_method == '2SLS':
        return 'series'
    if profile == 'benchmark2' and raw_method == 'Reg2SLS':
        return 'reg.'

    # The cached filename profile can include extra mc options such as skipfailedruns_True.
    if profile.endswith('_benchmark') and raw_method == '2SLS':
        return '2SLS'
    if profile.endswith('_benchmark2') and raw_method == '2SLS':
        return 'series'
    if profile.endswith('_benchmark2') and raw_method == 'Reg2SLS':
        return 'reg.'

    if 'RKHS' in raw_method:
        return raw_method
    if raw_method in {'AGMM2', 'AGMM2L2'}:
        return 'neural'
    return raw_method


def _expected_methods_from_configs(prefix: str):
    methods = []
    for path in sorted(CONFIG_DIR.glob(f'config_{prefix}_*.py')):
        profile = _profile_from_config(path, prefix)
        for raw_method in _method_keys_from_config(path):
            methods.append(_display_method(profile, raw_method))
    return _ordered_unique(BENCH_COLS + [m for m in methods if m not in BENCH_COLS])


def _method_columns(df, expected_cols):
    present = [] if df.empty else list(df['method'].dropna().unique())
    rkhs_expected = [m for m in expected_cols if 'RKHS' in m]
    rkhs_extra = sorted([m for m in present if 'RKHS' in m and m not in rkhs_expected])
    nonbench_extra = sorted([m for m in present if m not in BENCH_COLS and 'RKHS' not in m and m != 'neural'])
    neural_cols = ['neural'] if ('neural' in expected_cols or 'neural' in present) else []
    return _ordered_unique(BENCH_COLS + rkhs_expected + rkhs_extra + nonbench_extra + neural_cols)


NP_EXPECTED_METHOD_COLS = _expected_methods_from_configs('np')
SP_EXPECTED_METHOD_COLS = _expected_methods_from_configs('sp')
print('Expected NP columns:', NP_EXPECTED_METHOD_COLS)
print('Expected SP columns:', SP_EXPECTED_METHOD_COLS)


Expected NP columns: ['2SLS', 'series', 'reg.', 'ApproxRKHS2IV', 'ApproxRKHS2IVL2', 'neural', 'RKHS2IV', 'RKHS2IVL2']
Expected SP columns: ['2SLS', 'series', 'reg.', 'ApproxRKHS2IV', 'ApproxRKHS2IVL2', 'neural', 'RKHS2IV', 'RKHS2IVL2']


In [14]:
# -----------------------------
# 1) Nonparametrics table
# -----------------------------
NP_FN_RE = re.compile(r'_fn_(?P<fn>-?\d+)_')
NP_NEXP_RE = re.compile(r'^results_nexperiments_(?P<nexp>\d+)_')
NP_LEGACY_TAGS = ['2SLS_Reg2SLS', 'RKHS2L2', 'ApproxRKHS']


def _np_extract_tag(path: Path):
    stem = path.stem
    if '_gridtest_' in stem:
        tail = stem.split('_gridtest_', 1)[1]
        return tail.split('_', 1)[1] if '_' in tail else None

    known_tags = _ordered_unique(NP_LEGACY_TAGS + [m for m in NP_EXPECTED_METHOD_COLS if m not in {'series', 'reg.', 'neural'}] + ['AGMM2L2'])
    for tag in sorted(known_tags, key=len, reverse=True):
        if stem.endswith(f'_{tag}'):
            return tag
    return None


def _np_parse_meta(path: Path):
    fn_m = NP_FN_RE.search(path.name)
    nexp_m = NP_NEXP_RE.match(path.name)
    tag = _np_extract_tag(path)
    if fn_m is None or nexp_m is None or tag is None:
        return None

    return {
        'path': path,
        'nexp': int(nexp_m.group('nexp')),
        'fn': int(fn_m.group('fn')),
        'tag': tag,
        'mtime': path.stat().st_mtime,
    }


def _np_display_method(tag: str, method_key: str):
    if tag == '2SLS' and method_key == '2SLS':
        return '2SLS'
    if tag == '2SLS_Reg2SLS' and method_key == '2SLS':
        return 'series'
    if tag == '2SLS_Reg2SLS' and method_key == 'Reg2SLS':
        return 'reg.'
    if 'RKHS' in method_key:
        return method_key
    if 'RKHS' in tag:
        return tag
    if method_key in {'AGMM2L2', 'AGMM2'} or tag in {'AGMM2L2', 'AGMM2'}:
        return 'neural'
    return None


def _np_mse(pred, true):
    pred = np.asarray(pred)
    true = np.asarray(true)
    return float(np.mean((pred - true) ** 2))


def _upper_trim(arr, upper_q):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0 or upper_q is None:
        return arr
    cap = np.percentile(arr, float(upper_q))
    trimmed = arr[arr <= cap]
    return trimmed if trimmed.size > 0 else arr


np_records = []
np_skipped_files = []
for path in sorted(NP_DIR.glob('results_*.jbl')):
    meta = _np_parse_meta(path)
    if meta is None:
        np_skipped_files.append(path.name)
        continue

    runs = joblib.load(path)
    vals_by_method = {}
    for run in runs:
        if not isinstance(run, (tuple, list)) or len(run) != 2:
            continue
        param_estimates, true_params = run
        if not isinstance(param_estimates, dict) or not isinstance(true_params, dict):
            continue
        if 'dgp2' not in param_estimates or 'dgp2' not in true_params:
            continue

        method_dict = param_estimates['dgp2']
        true_target = true_params['dgp2']
        if not isinstance(method_dict, dict):
            continue

        for method_key, pred in method_dict.items():
            disp = _np_display_method(meta['tag'], method_key)
            if disp is None:
                continue
            mse_val = _np_mse(pred, true_target)
            if np.isfinite(mse_val):
                vals_by_method.setdefault(disp, []).append(mse_val)

    for method, vals in vals_by_method.items():
        arr = _upper_trim(vals, NP_TRIM_UPPER_Q)
        if arr.size == 0:
            continue
        np_records.append({
            'fn': meta['fn'],
            'method': method,
            **_mean_sd_se(arr),
            'n_runs': int(arr.size),
            'nexp': meta['nexp'],
            'source_file': path.name,
            'mtime': meta['mtime'],
        })

np_df = pd.DataFrame(np_records)
if not np_df.empty:
    np_df = np_df.sort_values(['fn', 'method', 'nexp', 'mtime'])
    np_df = np_df.groupby(['fn', 'method'], as_index=False).tail(1)
    np_df = np_df.drop(columns=['mtime']).sort_values(['fn', 'method']).reset_index(drop=True)

np_method_cols = _method_columns(np_df, NP_EXPECTED_METHOD_COLS)

np_table = pd.DataFrame(index=[x[0] for x in NP_FN_ORDER], columns=np_method_cols, dtype=object)
for fn_name, fn_code in NP_FN_ORDER:
    for method in np_method_cols:
        r = np_df[(np_df['fn'] == fn_code) & (np_df['method'] == method)] if not np_df.empty else pd.DataFrame()
        np_table.loc[fn_name, method] = '' if r.empty else _fmt_mean_se(r.iloc[0]['mean'], r.iloc[0]['se'])

display(Markdown('## Table 1: Nonparametrics'))
display(np_table)
if np_skipped_files:
    display(Markdown('**Skipped NP files that did not match the result filename pattern:** ' + ', '.join(np_skipped_files)))


## Table 1: Nonparametrics

,2SLS,series,reg.,ApproxRKHS2IV,ApproxRKHS2IVL2,RKHS2IV,RKHS2IVL2,neural
linear,0.004906 (0.0001063),9882 (890.5),0.5229 (0.01221),0.006314 (0.0001215),0.01239 (7.62e-05),0.006309 (0.0001216),0.01395 (5.071e-05),0.006491 (6.913e-05)
pwlinear,0.008595 (2.941e-05),1.111e+04 (1007),0.2039 (0.005031),0.00939 (3.9e-05),0.01162 (5.119e-05),0.009379 (3.889e-05),0.01029 (2.846e-05),0.01226 (7.932e-05)
sigmoid,0.005752 (4.07e-05),1.707e+04 (1499),0.07611 (0.0009868),0.007888 (7.645e-05),0.01208 (7.488e-05),0.007827 (7.545e-05),0.01425 (5.28e-05),0.0102 (0.0001019)
exponential,0.06297 (0.0006057),2570 (256.6),37.34 (1.598),0.05977 (0.0004354),0.02057 (0.0001024),0.05975 (0.0004365),0.02304 (8.421e-05),0.01969 (0.0001212)


In [15]:
# -----------------------------
# 2-5) Semiparametrics tables
# -----------------------------
SP_FILE_RE = re.compile(
    r'^results_nexperiments_(?P<nexp>\d+)_seed_(?P<seed>\d+)_(?P<profile>.+)_(?P<nsamp>\d+)_fn_(?P<fn>\d+)_(?P<method>[^.]+)\.jbl$'
)


def _sp_parse_meta(path: Path):
    m = SP_FILE_RE.match(path.name)
    if m is None:
        return None
    d = m.groupdict()
    return {
        'path': path,
        'nexp': int(d['nexp']),
        'profile': d['profile'],
        'fn': int(d['fn']),
        'raw_method': d['method'],
        'mtime': path.stat().st_mtime,
    }


def _sp_display_method(profile: str, raw_method: str):
    return _display_method(profile, raw_method)


def _sp_valid_ci(ci):
    try:
        if ci is None or len(ci) != 2:
            return None
        lo = float(np.asarray(ci[0]).reshape(-1)[0])
        hi = float(np.asarray(ci[1]).reshape(-1)[0])
        if np.isfinite(lo) and np.isfinite(hi):
            return lo, hi
    except Exception:
        pass
    return None


def _sp_summarize_runs(runs, true_param=SP_TRUE_PARAM):
    theta_vals, var_vals, cov_vals, len_vals = [], [], [], []

    for run in runs:
        if not isinstance(run, (tuple, list)) or len(run) < 3:
            continue
        try:
            theta = float(np.asarray(run[0]).reshape(-1)[0])
            var = float(np.asarray(run[1]).reshape(-1)[0])
        except Exception:
            continue

        if np.isfinite(theta):
            theta_vals.append(theta)
        if np.isfinite(var):
            var_vals.append(var)

        ci = _sp_valid_ci(run[2])
        if ci is not None:
            lo, hi = ci
            cov_vals.append(float((hi >= true_param) and (lo <= true_param)))
            len_vals.append(hi - lo)

    theta_arr = np.asarray(theta_vals, dtype=float)
    var_arr = np.asarray(var_vals, dtype=float)
    cov_arr = np.asarray(cov_vals, dtype=float)
    len_arr = np.asarray(len_vals, dtype=float)

    def _sp_upper_trim(arr, upper_q):
        arr = np.asarray(arr, dtype=float)
        arr = arr[np.isfinite(arr)]
        if arr.size == 0 or upper_q is None:
            return arr
        cap = np.percentile(arr, float(upper_q))
        trimmed = arr[arr <= cap]
        return trimmed if trimmed.size > 0 else arr

    bias_arr = theta_arr - true_param if theta_arr.size > 0 else theta_arr
    bias_trim = _sp_upper_trim(bias_arr, SP_BIAS_TRIM_UPPER_Q)
    var_trim = _sp_upper_trim(var_arr, SP_VARIANCE_TRIM_UPPER_Q)

    return {
        'bias': _mean_sd_se(bias_trim),
        'variance': _mean_sd_se(var_trim),
        'coverage': _mean_sd_se_coverage(cov_arr),
        'length': _mean_sd_se(len_arr),
    }


sp_best = {}
sp_skipped_files = []
for path in sorted(SP_DIR.glob('results_*.jbl')):
    meta = _sp_parse_meta(path)
    if meta is None:
        sp_skipped_files.append(path.name)
        continue
    disp = _sp_display_method(meta['profile'], meta['raw_method'])
    if disp is None:
        continue
    key = (meta['fn'], disp)
    old = sp_best.get(key)
    if old is None or (meta['nexp'] > old['nexp']) or (meta['nexp'] == old['nexp'] and meta['mtime'] > old['mtime']):
        sp_best[key] = meta

sp_rows = []
for (fn, disp), meta in sorted(sp_best.items()):
    runs = joblib.load(meta['path'])
    s = _sp_summarize_runs(runs, true_param=SP_TRUE_PARAM)
    row = {'fn': fn, 'method': disp, 'source_file': meta['path'].name, 'nexp': meta['nexp']}
    for metric in ['bias', 'variance', 'coverage', 'length']:
        row[f'{metric}_mean'] = s[metric]['mean']
        row[f'{metric}_sd'] = s[metric]['sd']
        row[f'{metric}_se'] = s[metric]['se']
        row[f'{metric}_n'] = s[metric]['n']
    sp_rows.append(row)

sp_df = pd.DataFrame(sp_rows)
if not sp_df.empty:
    sp_df = sp_df.sort_values(['fn', 'method']).reset_index(drop=True)
sp_method_cols = _method_columns(sp_df, SP_EXPECTED_METHOD_COLS)

for fn_name, fn_code in SP_FN_ORDER:
    sub = sp_df[sp_df['fn'] == fn_code] if not sp_df.empty else pd.DataFrame()
    tbl = pd.DataFrame(index=['bias', 'variance', 'coverage', 'length'], columns=sp_method_cols, dtype=object)
    for method in sp_method_cols:
        r = sub[sub['method'] == method] if not sub.empty else pd.DataFrame()
        if r.empty:
            for metric in ['bias', 'variance', 'coverage', 'length']:
                tbl.loc[metric, method] = ''
            continue
        rr = r.iloc[0]
        for metric in ['bias', 'variance', 'coverage', 'length']:
            tbl.loc[metric, method] = _fmt_mean_se(rr[f'{metric}_mean'], rr[f'{metric}_se'])

    display(Markdown(f'## Table 2 : ({fn_name})'))
    display(tbl)

if sp_skipped_files:
    display(Markdown('**Skipped SP files that did not match the result filename pattern:** ' + ', '.join(sp_skipped_files)))


## Table 2 : (linear)

,2SLS,series,reg.,ApproxRKHS2IV,ApproxRKHS2IVL2,RKHS2IV,RKHS2IVL2,neural
bias,-0.0006806 (0.001634),-4.361e+05 (4.343e+05),-0.4611 (0.1629),0.000413 (0.001837),0.07449 (0.001459),0.001267 (0.001843),0.07778 (0.001478),-0.0004437 (0.001721)
variance,27.39 (0.03659),4.518e+10 (6.693e+09),281.6 (5.935),34.6 (0.1032),20.57 (0.01822),34.64 (0.1064),20.82 (0.0189),30.03 (0.04302)
coverage,0.9468 (0.003174),0.9394 (0.003374),0.9334 (0.003526),0.9458 (0.003202),0.8736 (0.004699),0.946 (0.003196),0.8672 (0.004799),0.9428 (0.003284)
length,0.459 (0.0003208),1.843e+07 (1.689e+07),3.516 (0.6188),0.5164 (0.0008896),0.3977 (0.0001837),0.5168 (0.0009128),0.4002 (0.0001913),0.4806 (0.000359)


## Table 2 : (pwlinear)

,2SLS,series,reg.,ApproxRKHS2IV,ApproxRKHS2IVL2,RKHS2IV,RKHS2IVL2,neural
bias,-0.02801 (0.001771),-6.082e+04 (2.86e+04),-0.00325 (0.001713),-0.02785 (0.002299),0.07237 (0.001485),-0.02608 (0.002299),0.06825 (0.001516),-0.02258 (0.001772)
variance,32.63 (0.06096),4.488e+11 (7.881e+10),27.6 (0.04424),53.16 (0.3189),21.53 (0.02),53.18 (0.3224),22.15 (0.02105),31.27 (0.04062)
coverage,0.9466 (0.00318),0.9186 (0.003867),0.9388 (0.00339),0.9386 (0.003395),0.8818 (0.004566),0.9414 (0.003322),0.8876 (0.004467),0.9366 (0.003446)
length,0.501 (0.0004925),3.953e+05 (1.421e+05),0.4609 (0.0003943),0.6348 (0.001977),0.4069 (0.0001958),0.6352 (0.002015),0.4128 (0.0002033),0.4903 (0.0003288)


## Table 2 : (sigmoid)

,2SLS,series,reg.,ApproxRKHS2IV,ApproxRKHS2IVL2,RKHS2IV,RKHS2IVL2,neural
bias,-0.02283 (0.00175),-5.664e+04 (4.489e+04),-0.006259 (0.001861),-0.02383 (0.001762),0.07158 (0.00148),-0.0236 (0.001763),0.05779 (0.00152),-0.03411 (0.001744)
variance,31.79 (0.05686),7.37e+10 (1.232e+10),34.96 (0.0934),31.75 (0.05798),21.19 (0.0195),31.74 (0.05771),21.89 (0.02062),26.91 (0.02985)
coverage,0.9488 (0.003117),0.9224 (0.003784),0.9428 (0.003284),0.945 (0.003224),0.8808 (0.004582),0.9454 (0.003213),0.8984 (0.004273),0.9194 (0.00385)
length,0.4945 (0.0004649),2.047e+05 (1.298e+05),0.5183 (0.0007302),0.4942 (0.0004841),0.4037 (0.0001927),0.4942 (0.0004939),0.4103 (0.0002),0.4549 (0.000261)


## Table 2 : (exponential)

,2SLS,series,reg.,ApproxRKHS2IV,ApproxRKHS2IVL2,RKHS2IV,RKHS2IVL2,neural
bias,-0.202 (0.1565),-2.152e+08 (2.092e+08),-1.373e+12 (9.196e+11),-86.77 (76.63),-21.19 (11.78),-433.1 (262),-6.499e+04 (6.327e+04),0.0239 (0.005543)
variance,3086 (396.7),4.675e+15 (6.959e+14),1.963e+27 (2.5e+26),3.45e+05 (4.717e+04),6.732e+05 (9.687e+04),1.654e+08 (2.362e+07),8.165e+10 (1.384e+10),77.42 (0.8931)
coverage,0.958 (0.002837),0.9822 (0.00187),0.8808 (0.004582),0.9644 (0.00262),0.956 (0.0029),0.979 (0.002028),0.9762 (0.002156),0.9494 (0.0031)
length,11.13 (3.491),9.322e+08 (8.155e+08),1.48e+13 (6.279e+12),373.7 (299.6),189.3 (64.37),2566 (1071),2.7e+05 (2.443e+05),0.7957 (0.01873)


In [16]:
# -----------------------------
# LaTeX export for all 5 tables
# -----------------------------
def _fmt_latex_number(x, digits=4):
    if pd.isna(x):
        return ''
    x = float(x)
    ax = abs(x)
    if ax == 0:
        return '0'
    if ax >= 1e3 or ax < 1e-2:
        s = f'{x:.{digits - 1}e}'
        mant, exp = s.split('e')
        mant = float(mant)
        exp = int(exp)
        if abs(mant - round(mant)) < 1e-10:
            mant_str = str(int(round(mant)))
        else:
            mant_str = f'{mant:.{digits}g}'
        return f'{mant_str}$\\times 10^{{{exp}}}$'
    return f'{x:.{digits}g}'

def _fmt_latex_mean_se(mean_val, se_val):
    if pd.isna(mean_val):
        return ''
    mean = _fmt_latex_number(mean_val)
    if pd.isna(se_val):
        return mean
    return f'{mean} ({_fmt_latex_number(se_val)})'

def _latex_header(method_cols, bench_cols):
    n_methods = len(method_cols)
    bench_n = len([m for m in method_cols if m in bench_cols])
    prop_n = n_methods - bench_n

    colspec = 'c' * (n_methods + 1)
    lines = [rf'\begin{{tabular}}{{{colspec}}}']

    if prop_n > 0:
        lines.append(rf'    & \multicolumn{{{bench_n}}}{{c}}{{Benchmarks}}  & \multicolumn{{{prop_n}}}{{c}}{{Proposals}}  \tabularnewline')
        lines.append(rf'    \cmidrule(lr){{2-{1+bench_n}}}\cmidrule(lr){{{2+bench_n}-{1+n_methods}}}')
    else:
        lines.append(rf'    & \multicolumn{{{bench_n}}}{{c}}{{Benchmarks}}  \tabularnewline')
        lines.append(rf'    \cmidrule(lr){{2-{1+bench_n}}}')

    lines.append('     & ' + ' & '.join(method_cols) + r' \tabularnewline')
    lines.append(r'    \hline')
    return lines

def _latex_footer(lines):
    lines.append(r'    \hline')
    lines.append(r'\end{tabular}')
    return lines

# 1) Nonparametric block
np_lines = _latex_header(np_method_cols, BENCH_COLS)
for fn_name, fn_code in NP_FN_ORDER:
    vals = []
    for method in np_method_cols:
        r = np_df[(np_df['fn'] == fn_code) & (np_df['method'] == method)]
        vals.append('' if r.empty else _fmt_latex_mean_se(r.iloc[0]['mean'], r.iloc[0]['se']))
    np_lines.append(f'    {fn_name} & ' + ' & '.join(vals) + r' \tabularnewline')
np_lines = _latex_footer(np_lines)

# 2-5) Semiparametric blocks
sp_blocks = []
panel_titles = {0: '(a) Linear DGP', 1: '(b) Piecewise linear DGP', 2: '(c) Sigmoid DGP', 4: '(d) Exponential DGP'}
for _fn_name, _fn_code in SP_FN_ORDER:
    sub = sp_df[sp_df['fn'] == _fn_code]
    if sub.empty:
        continue
    lines = _latex_header(sp_method_cols, BENCH_COLS)
    for metric in ['bias', 'variance', 'coverage', 'length']:
        vals = []
        for method in sp_method_cols:
            r = sub[sub['method'] == method]
            vals.append(
                '' if r.empty
                else _fmt_latex_mean_se(r.iloc[0][f'{metric}_mean'], r.iloc[0][f'{metric}_se'])
            )
        lines.append(f'    {metric} & ' + ' & '.join(vals) + r' \tabularnewline')
    lines = _latex_footer(lines)
    sp_blocks.append((panel_titles.get(_fn_code, _fn_name), '\n'.join(lines)))

out_lines = []
out_lines.append('% Entries are mean (SE)')
out_lines.append('% Table 1: Nonparametrics')
out_lines.append('\n'.join(np_lines))
out_lines.append('')
for title, block in sp_blocks:
    out_lines.append(f'% {title}')
    out_lines.append(block)
    out_lines.append('')

latex_text = '\n'.join(out_lines).strip() + '\n'
out_path = TABLE_TEX_PATH
out_path.write_text(latex_text, encoding='utf-8')
print('Saved LaTeX:', out_path)
print('--- Preview ---')
print('\n'.join(latex_text.splitlines()[:60]))

Saved LaTeX: /Users/isaacm/Dropbox/repos/NNPIV_JASA/output/tables.tex
--- Preview ---
% Entries are mean (SE)
% Table 1: Nonparametrics
\begin{tabular}{ccccccccc}
    & \multicolumn{3}{c}{Benchmarks}  & \multicolumn{5}{c}{Proposals}  \tabularnewline
    \cmidrule(lr){2-4}\cmidrule(lr){5-9}
     & 2SLS & series & reg. & ApproxRKHS2IV & ApproxRKHS2IVL2 & RKHS2IV & RKHS2IVL2 & neural \tabularnewline
    \hline
    linear & 4.906$\times 10^{-3}$ (1.063$\times 10^{-4}$) & 9.882$\times 10^{3}$ (890.5) & 0.5229 (0.01221) & 6.314$\times 10^{-3}$ (1.215$\times 10^{-4}$) & 0.01239 (7.62$\times 10^{-5}$) & 6.309$\times 10^{-3}$ (1.216$\times 10^{-4}$) & 0.01395 (5.071$\times 10^{-5}$) & 6.491$\times 10^{-3}$ (6.913$\times 10^{-5}$) \tabularnewline
    pwlinear & 8.595$\times 10^{-3}$ (2.941$\times 10^{-5}$) & 1.111$\times 10^{4}$ (1.007$\times 10^{3}$) & 0.2039 (5.031$\times 10^{-3}$) & 9.39$\times 10^{-3}$ (3.9$\times 10^{-5}$) & 0.01162 (5.119$\times 10^{-5}$) & 9.379$\times 10^{-3}$ (3.889$\ti